In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

c:\Users\Ramazan\projects\turkish-ner-berturk\ner_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ner_model_path = "./saved_berturk_ner"
tokenizer = AutoTokenizer.from_pretrained(ner_model_path)
model = AutoModelForTokenClassification.from_pretrained(ner_model_path)

ner_pipeline = pipeline(
    "token-classification", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple"
)

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Modeller başarıyla yüklendi!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4780.81it/s]
c:\Users\Ramazan\projects\turkish-ner-berturk\ner_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ramazan\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnin

Modeller başarıyla yüklendi!


In [3]:
def extract_ner_metadata(sentence):
    """
    Cümleyi NER modeline sokar ve ChromaDB'ye uygun metadata formatında döndürür.
    """
    predictions = ner_pipeline(sentence)
    
    metadata = {"PERSON": [], "LOCATION": [], "ORGANIZATION": []}
    label_mapping = {"PER": "PERSON", "LOC": "LOCATION", "ORG": "ORGANIZATION"}
    
    for entity in predictions:
        ent_group = entity.get('entity_group', entity.get('entity')).split('-')[-1]
        metadata_key = label_mapping.get(ent_group)
        word = entity['word']
        
        if metadata_key:
            metadata[metadata_key].append(word)
            
    return {
        "PERSON": ", ".join(metadata["PERSON"]) if metadata["PERSON"] else "Yok",
        "LOCATION": ", ".join(metadata["LOCATION"]) if metadata["LOCATION"] else "Yok",
        "ORGANIZATION": ", ".join(metadata["ORGANIZATION"]) if metadata["ORGANIZATION"] else "Yok"
    }

ornek_cumle = "Apple Türkiye, İstanbul'da yeni bir ofis açacağını duyurdu."
print("Örnek Metadata:", extract_ner_metadata(ornek_cumle))

Örnek Metadata: {'PERSON': 'Yok', 'LOCATION': 'İstanbul', 'ORGANIZATION': 'Apple Türkiye'}


In [ ]:
client = chromadb.Client()

try:
    client.delete_collection(name="ner_vektor_db")
except Exception:  
    pass

collection = client.create_collection(name="ner_vektor_db")

sentences = [
    "Ahmet Yılmaz bugün uçakla İzmir'e gitti.",
    "Apple Türkiye, İstanbul'da yeni bir ofis açacağını duyurdu.",
    "Mustafa Kemal Atatürk, 1923 yılında Ankara'da cumhuriyeti ilan etti.",
    "Galatasaray, Şükrü Saracoğlu Stadyumu'nda Fenerbahçe ile karşılaştı.",
    "Sağlık Bakanı Fahrettin Koca, yarın saat 14:00'te açıklama yapacak.",
    "Trendyol, yeni operasyon merkezini Kocaeli'nin Gebze ilçesinde açtı.",
    "Elon Musk, SpaceX şirketinin merkezini Teksas'a taşıma kararı aldı."
]

print("Veriler veritabanına ekleniyor...\n")

for i, text in enumerate(sentences):
    vector = embedder.encode(text).tolist()
    
    metadata = extract_ner_metadata(text)
    
    doc_id = f"doc_{i}"
    
    collection.add(
        ids=[doc_id],
        embeddings=[vector],
        metadatas=[metadata],
        documents=[text]
    )
    print(f"Eklendi: {text[:40]}... | Metadata: {metadata}")

print("\nTüm veriler Vector DB'ye başarıyla kaydedildi!")

Veriler veritabanına ekleniyor...

Eklendi: Ahmet Yılmaz bugün uçakla İzmir'e gitti.... | Metadata: {'PERSON': 'Ahmet Yılmaz', 'LOCATION': 'İzmir', 'ORGANIZATION': 'Yok'}
Eklendi: Apple Türkiye, İstanbul'da yeni bir ofis... | Metadata: {'PERSON': 'Yok', 'LOCATION': 'İstanbul', 'ORGANIZATION': 'Apple Türkiye'}
Eklendi: Mustafa Kemal Atatürk, 1923 yılında Anka... | Metadata: {'PERSON': 'Mustafa Kemal Atatürk', 'LOCATION': 'Ankara', 'ORGANIZATION': 'Yok'}
Eklendi: Galatasaray, Şükrü Saracoğlu Stadyumu'nd... | Metadata: {'PERSON': 'Yok', 'LOCATION': 'Yok', 'ORGANIZATION': 'Galatasaray, Şükrü Saracoğlu Stadyumu, Fenerbahçe'}
Eklendi: Sağlık Bakanı Fahrettin Koca, yarın saat... | Metadata: {'PERSON': 'Fahrettin Koca', 'LOCATION': 'Yok', 'ORGANIZATION': 'Yok'}
Eklendi: Trendyol, yeni operasyon merkezini Kocae... | Metadata: {'PERSON': 'Yok', 'LOCATION': 'Kocaeli, Gebze', 'ORGANIZATION': 'Trendyol'}
Eklendi: Elon Musk, SpaceX şirketinin merkezini T... | Metadata: {'PERSON': 'Elon Musk', 'LOCAT

In [ ]:
query_text = "Büyük teknoloji firmalarının yatırımları ve ofis açılışları"

query_vector = embedder.encode(query_text).tolist()

print(f"\n🔍 Arama Sorgusu: '{query_text}'")

print("\n--- 1. Tüm Veritabanında Anlamsal Arama ---")
results = collection.query(
    query_embeddings=[query_vector],
    n_results=2 
)

for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"Cümle: {doc}")
    print(f"Bulunan Varlıklar: {meta}\n")


print("\n--- 2. Metadata Filtreli Arama (Sadece İçinde ORG Geçenler) ---")
filtered_results = collection.query(
    query_embeddings=[query_vector],
    n_results=2,
    where={"ORGANIZATION": {"$ne": "Yok"}} 
)

for doc, meta in zip(filtered_results['documents'][0], filtered_results['metadatas'][0]):
    print(f"Cümle: {doc}")
    print(f"Kurum/Firma: {meta['ORGANIZATION']}\n")


🔍 Arama Sorgusu: 'Büyük teknoloji firmalarının yatırımları ve ofis açılışları'

--- 1. Tüm Veritabanında Anlamsal Arama ---
Cümle: Trendyol, yeni operasyon merkezini Kocaeli'nin Gebze ilçesinde açtı.
Bulunan Varlıklar: {'ORGANIZATION': 'Trendyol', 'PERSON': 'Yok', 'LOCATION': 'Kocaeli, Gebze'}

Cümle: Galatasaray, Şükrü Saracoğlu Stadyumu'nda Fenerbahçe ile karşılaştı.
Bulunan Varlıklar: {'LOCATION': 'Yok', 'ORGANIZATION': 'Galatasaray, Şükrü Saracoğlu Stadyumu, Fenerbahçe', 'PERSON': 'Yok'}


--- 2. Metadata Filtreli Arama (Sadece İçinde ORG Geçenler) ---
Cümle: Trendyol, yeni operasyon merkezini Kocaeli'nin Gebze ilçesinde açtı.
Kurum/Firma: Trendyol

Cümle: Galatasaray, Şükrü Saracoğlu Stadyumu'nda Fenerbahçe ile karşılaştı.
Kurum/Firma: Galatasaray, Şükrü Saracoğlu Stadyumu, Fenerbahçe

